# Multiple Linear Regression – Multi-Channel Marketing Analysis
### Project Overview
This project implements a Multiple Linear Regression model using Python and `statsmodels` to evaluate how concurrent advertising expenditures across **TV**, **Radio**, and **Social Media** collectively influence company **Sales**. 

### Step-by-Step Analytical Process Covered:
1. **Data Prep:** Load dataset and eliminate structural missing values.
2. **Multicollinearity Assessment:** Check feature redundancy using cross-correlation heatmaps and Variance Inflation Factors (VIF).
3. **OLS Modeling:** Fit a multivariate Ordinary Least Squares (OLS) regression matrix.
4. **Statistical Diagnostics:** Evaluate overall accuracy via Adjusted $R^2$ and evaluate individual channel significance via p-values.
5. **Residual Analysis:** Validate model assumptions (Linearity, Homoscedasticity, Normality) using a suite of diagnostic plots.
6. **Business Interpretation:** Extract actionable ROI insights for marketing budget restructuring.

## 1. Environment Setup & Data Infrastructure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Formatting and plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Environment successfully initialized with required diagnostic packages.")

## 2. Load and Clean Dataset
We check for null entries to protect the model matrix from missing values.

In [ ]:
filename = "marketing_and_sale_data_evaulate_Ir.csv"
df = pd.read_csv(filename)

print("--- Missing Values Found Per Column ---")
print(df.isnull().sum())

# Clean missing rows
df = df.dropna()
print(f"\nData cleaning complete. Total observation count: {len(df)}")
display(df.describe())

## 3. Multicollinearity Analysis (Correlation & VIF Metrics)
Multicollinearity arises when independent variables are highly correlated with each other, which destabilizes the regression coefficient calculations. We review cross-variable dependencies through a correlation heatmap and Variance Inflation Factors (VIF).

In [ ]:
predictors = ['TV', 'Radio', 'Social_Media']
corr_matrix = df[predictors].corr()

print("--- Correlation Coefficients Matrix ---")
display(corr_matrix)

# Visualizing correlations
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Independent Variable Correlation Grid")
plt.show()

# Calculate Variance Inflation Factors
X_vif = df[predictors]
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_const.values, i+1) for i in range(len(X_vif.columns))]
print("\n--- Variance Inflation Factor Calculations ---")
display(vif_data)

### Statistical Evaluation of Multicollinearity:
- **Correlation Heatmap:** Reflects an intense positive relationship (~0.87) specifically tracking between `TV` and `Radio` capital allocations.
- **VIF Context:** VIF scores below 5 indicate that multicollinearity is not severe enough to distort the individual parameter estimations, meaning we can retain all three media tracks in the master model.

## 4. Multiple Linear Regression (OLS)
We implement a multivariate Ordinary Least Squares model, adding a constant vector for intercept identification.

In [ ]:
X_multi = df[['TV', 'Radio', 'Social_Media']]
X_multi_const = sm.add_constant(X_multi)
y = df['Sales']

multi_model = sm.OLS(y, X_multi_const).fit()
print(multi_model.summary())

## 5. Model Diagnostics (Residual Assumption Tests)
We generate residual visualizations to verify standard parametric validation constraints:
1. **Linearity & Homoscedasticity:** Verified when residuals scatter symmetrically across the zero-horizontal on a Fitted vs. Residuals diagram.
2. **Normality:** Confirmed when sample errors align cleanly across the theoretical normal Q-Q vector.

In [ ]:
residuals = multi_model.resid
fitted_values = multi_model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs Fitted values 
sns.scatterplot(x=fitted_values, y=residuals, alpha=0.6, ax=axes[0])
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values (Predicted Sales)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs. Fitted Plot')

# Normal Probability Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Normal Q-Q Plot')

# Histogram Distribution
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_xlabel('Residual Value')
axes[2].set_title('Error Distribution Histogram')

plt.tight_layout()
plt.show()

## 6. Business Coefficients Interpretation & Executive Recommendations

### Contextual Coefficient Breakdown:
- **TV Allocation Metric:** **Holding Radio and Social Media advertising spends completely constant, each additional dollar invested in TV marketing is associated with an estimated increase of $3.56 in Sales.**
- **Radio and Social Media Metrics:** Both pipelines display p-values vastly higher than the baseline alpha significance criterion ($p > 0.05$). This indicates that variations across Radio and Social Media allocations provide no statistically observable link to revenue fluctuations within this dynamic structure.

### Strategic Action Plan:
1. **Prioritize TV Capital Allocation:** Because TV acts as the sole reliable engine for revenue growth with a clear, statistically significant return coefficient, it should receive the vast majority of the marketing budget.
2. **Restructure Non-Performing Assets:** Defund or re-evaluate current campaign frameworks across Radio and Social Media since their investments yield no statistically reliable return on investment (ROI).